In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import requests, base64, os, datetime

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
headers = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

In [13]:
def parse_repo(url: str) -> tuple:
    url = url.strip().rstrip("/")
    if "github.com" in url:
        parts = url.split("github.com/")[-1].split("/")
    else:
        parts = url.split("/")
    return parts[0], parts[1]


@tool
def get_repo_info(repo_url: str) -> str:
    """Get general info about a GitHub repo (description, stars, forks, language, topics)."""
    try:
        owner, repo = parse_repo(repo_url)
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}", headers=headers)
        if r.status_code != 200:
            return f"Error: {r.status_code} - {r.json().get('message', '')}"
        d = r.json()
        license_info = d.get('license') or {}
        return (
            f"Name: {d.get('full_name', 'N/A')}\n"
            f"Description: {d.get('description', 'No description')}\n"
            f"Language: {d.get('language', 'N/A')}\n"
            f"Stars: {d.get('stargazers_count', 0)}\n"
            f"Forks: {d.get('forks_count', 0)}\n"
            f"Open Issues: {d.get('open_issues_count', 0)}\n"
            f"Topics: {', '.join(d.get('topics') or [])}\n"
            f"Last Updated: {d.get('updated_at', 'N/A')}\n"
            f"License: {license_info.get('name', 'None')}\n"
            f"Homepage: {d.get('homepage', 'None')}"
        )
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_readme(repo_url: str) -> str:
    """Fetch the README content of a GitHub repo."""
    try:
        owner, repo = parse_repo(repo_url)
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}/readme", headers=headers)
        if r.status_code != 200:
            return "No README found."
        content = base64.b64decode(r.json()["content"]).decode("utf-8", errors="ignore")
        return content[:1500] + ("\n... (truncated)" if len(content) > 1500 else "")
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_file_structure(repo_url: str) -> str:
    """Get the top-level file and folder structure of a GitHub repo."""
    try:
        owner, repo = parse_repo(repo_url)
        r = requests.get(f"https://api.github.com/repos/{owner}/{repo}/contents/", headers=headers)
        if r.status_code != 200:
            return f"Error: {r.status_code}"
        items = r.json()
        structure = []
        for item in items:
            icon = "📁" if item["type"] == "dir" else "📄"
            structure.append(f"{icon} {item['name']}")
        return "\n".join(structure)
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_recent_commits(repo_url: str) -> str:
    """Get the last 5 commits of a GitHub repo."""
    try:
        owner, repo = parse_repo(repo_url)
        r = requests.get(
            f"https://api.github.com/repos/{owner}/{repo}/commits?per_page=5",
            headers=headers
        )
        if r.status_code != 200:
            return f"Error: {r.status_code}"
        result = []
        for c in r.json():
            msg = c["commit"]["message"].split("\n")[0]
            author = c["commit"]["author"]["name"]
            date = c["commit"]["author"]["date"][:10]
            result.append(f"[{date}] {author}: {msg}")
        return "\n".join(result)
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_languages(repo_url: str) -> str:
    """Get the programming languages breakdown of a GitHub repo."""
    try:
        owner, repo = parse_repo(repo_url)
        r = requests.get(
            f"https://api.github.com/repos/{owner}/{repo}/languages",
            headers=headers
        )
        if r.status_code != 200:
            return f"Error: {r.status_code}"
        langs = r.json()
        total = sum(langs.values())
        result = []
        for lang, b in sorted(langs.items(), key=lambda x: -x[1]):
            result.append(f"{lang}: {(b/total)*100:.1f}%")
        return "\n".join(result) if result else "No language data."
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def save_report(content: str) -> str:
    """Save the final analysis report to a file."""
    os.makedirs("../reports", exist_ok=True)
    filename = f"../reports/repo_analysis_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Report saved to {filename}"


tools = [
    get_repo_info,
    get_readme,
    get_file_structure,
    get_recent_commits,
    get_languages,
    save_report
]


In [15]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

system_prompt = """You are an expert code analyst. When given a GitHub repo URL, 
analyze it thoroughly using all available tools and produce a structured report covering:
- What the project does
- Tech stack and languages
- Project structure
- Recent activity
- How to get started
- Strengths and suggestions for improvement

Always save the final report using the save_report tool."""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)


C:\Users\USER\AppData\Local\Temp\ipykernel_23828\4086865176.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [18]:
repo_url = "https://github.com/ImadNait/PDF-Research-Assistant"

info       = get_repo_info.invoke(repo_url)
readme     = get_readme.invoke(repo_url)
structure  = get_file_structure.invoke(repo_url)
commits    = get_recent_commits.invoke(repo_url)
languages  = get_languages.invoke(repo_url)

summary_prompt = f"""
You are a code analyst. Based on the data below, write a clean structured report.

REPO INFO:
{info}

README:
{readme}

FILE STRUCTURE:
{structure}

RECENT COMMITS:
{commits}

LANGUAGES:
{languages}

Write a report covering:
1. What the project does
2. Tech stack
3. Project structure
4. Recent activity
5. How to get started
6. Strengths and suggestions
"""

response = llm.invoke(summary_prompt)

print("REPO ANALYSIS REPORT")
print("═" * 60)
print(response.content)

save_report.invoke(response.content)




REPO ANALYSIS REPORT
════════════════════════════════════════════════════════════
**Project Report: PDF Research Assistant**

### 1. Project Overview

The PDF Research Assistant is an AI-powered tool that enables users to ask questions about documents and receive instant answers with source citations. The project utilizes Retrieval-Augmented Generation (RAG) to provide semantic search capabilities across multiple PDF documents. The assistant can also provide optional AI-powered answers with Ollama.

### 2. Tech Stack

The project is built using the following technologies:

* **Language:** Python (100.0%)
* **Libraries and Frameworks:** LangChain, ChromaDB, and Ollama (optional)
* **Database:** ChromaDB (a persistent vector database)

### 3. Project Structure

The project has the following structure:

* **Root Directory:** Contains the `README.md` file, `pdf_db` directory, `pdf_rag.py` script, and `pdfs` directory
* **.vscode Directory:** Contains Visual Studio Code configuration files


'Report saved to ../reports/repo_analysis_20260612_120510.txt'